In [ ]:
import random
import re
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import time

# =================================================================================
# ✨ 데이터셋 개요: [meal-bbang/Korean_message]
# ✨ 의미: 한국어 스팸/피싱 메시지 분류 학습 데이터셋입니다.
# ✨ 목표: 이 스크립트는 초보자가 텍스트 분류(Text Classification)의 개념을 익히고,
#        실제 스팸 메시지의 특징을 데이터 분석을 통해 직관적으로 파악해 보는 실습입니다.
# ✨ Labels: 1 = 일상 메시지 (Normal), 2 = 피싱/스미싱 메시지 (Spam)
# =================================================================================

# --- 설정 변수 ---
DATASET_NAME = "meal-bbang/Korean_message"
SAMPLE_COUNT = 100  # 너무 많은 데이터를 한 번에 불러오는 것보다, 적은 샘플로도 충분히 재미있는 분석을 할 수 있습니다!
# -----------------

print("===============================================================")
print("💡 튜터링 시작! 안녕하세요! 스팸 메시지 탐정단에 오신 걸 환영해요! 🕵️‍♀️")
print("우리가 오늘 탐험할 데이터셋은 한국어 스팸 분류 데이터입니다.")
print("단순히 코드를 돌리는 것을 넘어, '스팸'이 무엇인지 컴퓨터가 어떻게 배울 수 있는지 같이 탐험해 봐요!")
print("===============================================================")

# ---------------------------------------------------------------
# 🛠️ Step 1: 데이터셋 준비 및 로딩 (스트리밍 vs. 다운로드 전략)
# ---------------------------------------------------------------

print("\n⚙️ [Step 1] 데이터 로딩 준비...")

# 1. Config 이름 확인 (만능 코딩의 시작은 관찰부터!)
DATASET_ID = DATASET_NAME.split('/')[0] # 'meal-bbang'
try:
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0]
except Exception as e:
    print("ℹ️ 해당 데이터셋은 별도 Config 없이 기본 설정으로 진행합니다.")
    selected_config = None

# 2. 데이터 로딩 시도 (스트리밍을 먼저 시도해 보자!)
dataset = None
try:
    # 스트리밍 모드 (메모리 효율적!)
    print("🌟 스트리밍(streaming=True) 모드로 데이터셋을 연결합니다. (느린 네트워크 환경에 대비!)")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 스트리밍 연결 성공! 데이터를 메모리에 모두 올리지 않아도 되니 참 편해요.")

except Exception as e:
    # 스트리밍 실패 시 (간혹 발생할 수 있어요!)
    print(f"⚠️ 스트리밍 모드 연결 오류 발생 ({e}). 일반 다운로드 모드로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 일반 다운로드 모드로 데이터를 성공적으로 불러왔습니다. (좀 더 안정적이에요!)")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드 실패: {e_fallback}")
        exit()

# 3. 샘플 추출 (데이터 전체를 보는 건 시간 낭비! 상위 100개만 볼게요!)
print(f"\n✨ 상위 {SAMPLE_COUNT}개의 샘플만을 추출하여 실습에 사용할 샘플 리스트를 준비합니다.")

if hasattr(dataset, "take"):
    # 스트리밍 데이터셋인 경우 (IterableDataset)
    sample_data_iterator = dataset.take(SAMPLE_COUNT)
    # 데이터를 리스트로 한번에 받아와야 분석에 유리해요!
    sample_data_list = list(sample_data_iterator)
else:
    # 일반 데이터셋인 경우 (Dataset)
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

print(f"✅ 총 {len(sample_data_list)}개의 샘플 데이터를 준비했어요. 이제 놀아봅시다!")


# ---------------------------------------------------------------
# 🚀 Step 2: 데이터 분석 및 패턴 탐색 (스팸 메시지의 특징 찾기)
# ---------------------------------------------------------------
print("\n" + "="*80)
print("🔎 [Step 2] 데이터 분석: 스팸 vs. 일상어는 어떤 차이가 있을까요?")
print("===============================================================")

# 1. 클래스 비율 분석 (가장 중요한 통계부터!)
class_counts = Counter()
for sample in sample_data_list:
    # 'class'는 레이블입니다. 1: 정상, 2: 스팸
    class_counts[sample['class']] += 1

total_samples = len(sample_data_list)
print(f"\n📊 1. 데이터 클래스 분포 분석 ({total_samples}개 샘플 기준):")

normal_count = class_counts.get(1, 0)
spam_count = class_counts.get(2, 0)

print(f"  🟢 [클래스 1: 일상 메시지] Count: {normal_count}개 ({normal_count/total_samples:.2%})")
print(f"  🔴 [클래스 2: 피싱/스팸 메시지] Count: {spam_count}개 ({spam_count/total_samples:.2%})")

if spam_count > normal_count:
    print("✨ 와우! 샘플에서는 스팸 메시지가 조금 더 많이 보이는 것 같네요. 주의해야 할 것들이 많다는 뜻이겠죠?")
elif normal_count > spam_count:
    print("🧐 샘플이 정상 메시지 위주로 구성된 것 같아요. 스팸 유형을 더 살펴볼 필요가 있습니다!")
else:
    print("⚖️ 두 클래스가 비슷한 비율로 섞여있어요. 아주 좋은 학습 데이터셋입니다!")


# 2. 키워드 분석 (스팸의 공통점 찾기!)
print("\n📝 2. 핵심 키워드 패턴 탐색 (직관적 EDA):")

def extract_keywords(text):
    # 문장을 소문자화하고, 한글 외의 문자(숫자나 영어)만 추출하여 리스트로 만듭니다.
    text = text.lower()
    words = re.findall(r'[가-힣]{2,}', text) # 2글자 이상의 한국어만 추출
    return words

spam_keywords = []
normal_keywords = []

for sample in sample_data_list:
    content = sample['content']
    keywords = extract_keywords(content)
    
    if sample['class'] == 2: # 스팸 메시지
        spam_keywords.extend(keywords)
    else: # 일상 메시지 (class == 1)
        normal_keywords.extend(keywords)

# 빈도 분석
spam_counts = Counter(spam_keywords)
normal_counts = Counter(normal_keywords)

# 상위 N개 키워드 출력
def print_top_n(counter, class_name):
    print(f"  >>> {class_name} 메시지의 가장 자주 등장하는 단어 (상위 5):")
    top_5 = counter.most_common(5)
    if top_5:
        for word, count in top_5:
            print(f"    - '{word}': 등장 횟수 {count}번")
    else:
        print("    [분석된 키워드가 없습니다.]")

print_top_n(spam_counts, "🔴 스팸")
print_top_n(normal_counts, "🟢 일반")


# ---------------------------------------------------------------
# 💡 Step 3: 간이 스팸 탐지 시뮬레이터 (내가 AI 개발자라고 상상해 보기)
# ---------------------------------------------------------------
print("\n" + "="*80)
print("🔮 [Step 3] 스팸 탐지 시뮬레이터: 내가 AI 개발자라면? (미니 분류 체험)")
print("===============================================================")
print("실제 머신러닝 모델은 복잡하지만, 개념적으로 '특정 키워드'가 보이면 스팸일 확률이 높다고 판단합니다.")

# 스팸 특유의 키워드 목록을 사람이 직접 설정해봅니다.
spam_trigger_keywords = ['건강', '대출', '금전', '클릭', '경품', '주문']

def simple_spam_detector(text_message):
    """입력된 메시지에 스팸 추정 키워드가 포함되어 있는지 체크하는 함수"""
    
    # 1. 전처리: 소문자 및 공백 제거 (가장 기초적인 AI 처리!)
    processed_text = text_message.lower()
    
    score = 0
    found_keywords = []

    # 2. 키워드 점수 부여: 스팸 키워드가 많을수록 점수 상승!
    for keyword in spam_trigger_keywords:
        if keyword in processed_text:
            score += 1
            found_keywords.append(keyword)
            
    # 3. 임계값 설정 및 예측
    if score >= 2:
        return "🚨 스팸 (Spam)입니다! (높은 점수: 2점 이상)", found_keywords
    elif score >= 1:
        return "⚠️ 의심 메시지 (Warning). (경고: 키워드 발견)", found_keywords
    else:
        return "✅ 정상 (Normal)으로 추정됩니다.", []

# 사용자 상호작용 시뮬레이션
print("\n📣 저와 함께 몇 가지 가상 메시지를 테스트해보실까요? (메시지 내용을 적어보세요!)")

# 테스트 1: 명백한 스팸 메시지
test_msg_1 = "손님 귀하, 저희가 발송한 청첩장인데, 지금 바로 클릭해서 확인해주세요! 건강한 삶을 응원합니다."
result_1, triggers_1 = simple_spam_detector(test_msg_1)
print(f"\n[Test 1] 메시지: '{test_msg_1}'")
print(f"   👉 결과: {result_1}")
print(f"   📌 감지 근거 키워드: {', '.join(triggers_1)}")

# 테스트 2: 정상적인 일상 메시지
test_msg_2 = "오늘 저녁에 만날 건데, 혹시 시간 괜찮을까? 같이 영화 보러 갈래?"
result_2, triggers_2 = simple_spam_detector(test_msg_2)
print(f"\n[Test 2] 메시지: '{test_msg_2}'")
print(f"   👉 결과: {result_2}")
print(f"   📌 감지 근거 키워드: {', '.join(triggers_2)}")

# 테스트 3: '대출'과 '금전'이 포함된 의심 메시지 (경계 케이스)
test_msg_3 = "금리가 낮아진 대출 상품이 나왔습니다. 클릭 후 상담받으세요!"
result_3, triggers_3 = simple_spam_detector(test_msg_3)
print(f"\n[Test 3] 메시지: '{test_msg_3}'")
print(f"   👉 결과: {result_3}")
print(f"   📌 감지 근거 키워드: {', '.join(triggers_3)}")


print("\n\n==============================================================")
print("🥳 수고하셨습니다! 코딩과 분석을 통해 '스팸'의 패턴을 이해하는 것이 첫걸음이에요.")
print("다음 단계에서는 이 키워드 분석을 바탕으로, 실제 Scikit-learn 같은 라이브러리를 이용해 모델을 만들어 볼 수 있을 거예요! 파이팅!")
print("==============================================================")